In [87]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymysql

In [88]:
try:
    # DB 연결
    conn = pymysql.connect(
        host = 'localhost',
        user = 'root',
        passwd = '235700'
    )
    cur = conn.cursor()

    def sql_get(sql):
        cur.execute(sql)
        return cur.fetchall()

    def sql_set(sql):
        cur.execute(sql)
        conn.commit()

    # DB 만들기 (data_repo)
    sql_set('''DROP DATABASE IF EXISTS data_repo''')
    sql_set('''CREATE DATABASE data_repo''')
    sql_set('''USE data_repo''')

    # 파일에서 데이터 가져오기
    paths = 'data-files/iris.data'
    colums = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
    iris_df = pd.read_csv(paths, header=None, names=colums)

    # 테이블 만들기 (iris)
    sql_set('''CREATE TABLE IF NOT EXISTS iris(
                idx          INT         AUTO_INCREMENT,
                sepal_length FLOAT       NOT NULL,
                sepal_width  FLOAT       NOT NULL,
                petal_length FLOAT       NOT NULL,
                petal_width  FLOAT       NOT NULL,
                species      VARCHAR(30) NOT NULL,
                PRIMARY KEY (idx)
            )
            ''')

    # 데이터 삽입
    cur.executemany(f'''
                    INSERT INTO iris ({','.join(colums)})
                    VALUES (%s, %s, %s, %s, %s)
                    ''', iris_df.values.tolist())
    conn.commit()
except Exception as e:
    print(e)
    if conn:
        conn.rollback()
finally:
    # 연결 해제
    if cur:
        cur.close()
    if conn:
        conn.close()


In [89]:
try:
    # DB로 DF 만들기
    conn = pymysql.connect(
        host = 'localhost',
        user = 'root',
        passwd = '235700',
        database = 'data_repo'
    )
    cur = conn.cursor()
    def sql_get(sql):
        cur.execute(sql)
        return cur.fetchall()

    def sql_set(sql):
        cur.execute(sql)
        conn.commit()

    colums.insert(0, 'idx')
    iris_df2 = pd.DataFrame(sql_get(f'''SELECT {','.join(colums)} FROM iris'''), columns=colums)

except Exception as e:
    print(e)
    if conn:
        conn.rollback()
finally:
    # 연결 해제
    if cur:
        cur.close()
    if conn:
        conn.close()

iris_df2


,idx,sepal_length,sepal_width,petal_length,petal_width,species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...,...
145,146,6.7,3.0,5.2,2.3,Iris-virginica
146,147,6.3,2.5,5.0,1.9,Iris-virginica
147,148,6.5,3.0,5.2,2.0,Iris-virginica
148,149,6.2,3.4,5.4,2.3,Iris-virginica


In [90]:
# 읽은 데이터를 파일로 저장
paths = "data-files/iris_from_db.csv"

iris_df2.to_csv(paths, index=False, header=None)